# Actividad 3 | Aprendizaje supervisado y no supervisado | Dataset RetailRocket

## Objetivo 

Aplicar algoritmos de aprendizaje supervisado y no supervisado mediante PySpark para la resolución de problemas en análisis de datos, fomentando el desarrollo de habilidades prácticas en el manejo y procesamiento eficiente de grandes conjuntos de datos.

## Instrucciones

En esta actividad aprenderás a aplicar algoritmos de aprendizaje supervisado y no supervisado implementados en PySpark, identificando para ello cada una de las etapas necesarias para poder ejecutar con éxito dichos algoritmos. En particular, se espera que apliques con éxito un algoritmo de aprendizaje supervisado (DecisionTree, RandomForest, GBTClassifier, Multilayer Perceptron, entre otros) y uno de aprendizaje no supervisado (K means, GaussianMixture, PIC, entre otros) a partir de las implementaciones disponibles en PySpark, aplicando las etapas necesarias y suficientes de preparación de los datos de entrada.

Para lograrlo, se sugiere seguir los siguientes pasos:

1. Introducción teórica
   - Se deberá de documentar de forma breve los conceptos de aprendizaje supervisado, no supervisado, así como los algoritmos más representativos que se identifican en la literatura de cada tipo de aprendizaje, además de identificar aquellos que están disponible a través de PySpark.
2. Selección de los datos
   - Para esta actividad, se propone recolectar una muestra de dimensión contenida (para evitar que los tiempos de procesamiento sean altos) a partir de la base de datos que estás trabajando en tu proyecto. Para ello y tomando como base la actividad previa en la cual has implementado códigos que permiten obtener particiones de la base de datos global D que cumplen con los criterios de las variables de caracterización identificadas, se propone que recuperes un número limitado de instancias de cada partición (aplicando la técnica de muestreo que propusiste en el Módulo 3, Proyecto Base de datos de Big Data, paso 4), lo que te permitirá construir una muestra M a partir de la unión de las instancias que se recuperan de este proceso
4. Preparación de los datos
   - En esta etapa, se deberán de aplicar estrategias de corrección sobre los datos que integran a la muestra M que se ha preparado en el paso previo, de tal forma que de deje un conjunto M listo para ser procesado por los algoritmos de aprendizaje a aplicar. Para ello se deben de considerar pasos como: corrección de registros / columnas con valores nulos, identificación de valores atípicos, transformación de los tipos de datos, etc. Con lo anterior, se tendrá una muestra M pre-procesada.
6. Preparación del conjunto de entrenamiento y prueba
   - Para esta etapa, la muestra M será divida en un conjunto de entrenamiento y prueba. Para ello, deberás proponer una técnica de muestreo que te permita construir el conjunto de entrenamiento y prueba minimizando el riesgo de inyección de sesgos. Ten en cuenta que, para este punto, deberás de tener en claro el porcentaje de división a utilizar, el cual se deberá de justificar.
8. Construcción de modelos de aprendizaje supervisado y no supervisado
   - Para este punto realizarás dos experimentos separados, dónde se aplicará un algoritmo de aprendizaje supervisado y uno de aprendizaje no supervisado sobre la muestra M. Para el caso de aprendizaje supervisado, se deberá de identificar cuál es la variable objetivo (columna) de aprendizaje, mientras que, para el caso de aprendizaje no supervisado, se debe de seleccionar todas las columnas que se desean considerar como características bajo las cuales se realizará el proceso de agrupamiento. Usando las implementaciones correspondientes de PySpark, se deberá de ejecutar el aprendizaje correspondiente a partir de la invocación de las funciones respectivas. Para este ejercicio, se deberá seleccionar un criterio básico para medir la calidad del resultado obtenido, dependiendo de cada tipo de aprendizaje implementado. La elección quedará a juicio de cada estudiante.
  

## Especificaciones de entrega

1. Introducción: sección dónde se da respuesta al punto 1
2. Selección de los datos: código, con documentación, donde se implementan las etapas para la construcción de muestra inicial M (punto 2)
3. Preparación de los datos: etapa donde se pre-procesa la muestra M, para corregir formatos e inconsistencias de cualquier índole que tenga la muestra original. Se deberá de documentar los pasos que se implementen para resolver esta etapa.
4. Preparación del conjunto de entrenamiento y prueba: código en la cual se construye el conjunto de entrenamiento y prueba para la experimentación, debidamente documentado el código.
5. Construcción de modelos de aprendizaje supervisado y no supervisado: se recomienda dividir en dos subsecciones, una donde se muestre como se entrena un algoritmo de aprendizaje supervisado, y otra para el entrenamiento de un algoritmo no supervisado. Se debe de documentar el código implementado con una breve discusión de resultados.

## 1. Introducción teórica

El **aprendizaje supervisado** usa datos históricos que tienen una variable objetivo. El algoritmo aprende una relación entre variables predictoras y una respuesta esperada. En problemas de clasificación, la salida es una clase; en problemas de regresión, la salida es un valor numérico. En PySpark MLlib se encuentran algoritmos supervisados como `DecisionTreeClassifier`, `RandomForestClassifier`, `GBTClassifier`, `LogisticRegression`, `LinearRegression` y `MultilayerPerceptronClassifier`.

El **aprendizaje no supervisado** trabaja con datos sin etiqueta. Su propósito es descubrir la relacion interna, grupos, patrones o representaciones latentes. En PySpark se encuentran métodos como `KMeans`, `BisectingKMeans`, `GaussianMixture`, `LDA` y `PowerIterationClustering`.

En esta entrega se implementan dos experimentos:

- **Experimento supervisado:** `RandomForestClassifier`. Se elige porque maneja relaciones no lineales, es robusto ante variables numéricas con escalas distintas y permite revisar importancia de variables. La variable objetivo será `label`, donde `0 = view` y `1 = addtocart o transaction`.
- **Experimento no supervisado:** `KMeans`. Se elige porque es un método base de agrupamiento, eficiente en grandes volúmenes de datos y disponible de forma directa en PySpark. Su calidad se medirá con el KPI de **silhouette**.

Se evita usar identificadores (`visitorid`, `itemid`) como variables predictoras directas para reducir memorización. También se excluye `transactionid`, porque su presencia está asociada directamente a compras y produciría fuga de información en el modelo supervisado.


## 2. Selección de los datos

Se parte del archivo principal `events.csv` del dataset RetailRocket. A partir de este archivo se construye la muestra **M** usando la misma lógica de la actividad previa:

- `P1`: eventos `view` de usuarios con actividad baja.
- `P2`: eventos `view` de usuarios con actividad media.
- `P3`: eventos `view` de usuarios con actividad alta.
- `P4`: eventos `addtocart`.
- `P5`: eventos `transaction`.

La muestra conserva una fracción pequeña de visualizaciones y una proporción mayor de eventos minoritarios, con censo completo de transacciones. Esto reduce el tiempo de procesamiento y preserva señales importantes de intención de compra.


In [162]:
%pip install -q pyspark kagglehub

Note: you may need to restart the kernel to use updated packages.


In [163]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [164]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("RetailRocket-Actividad-3")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

SEED = 42


In [165]:
DATASET_PATH = kagglehub.dataset_download("retailrocket/ecommerce-dataset") # Dataset descargado desde Kaggle

print("Ruta del dataset:", DATASET_PATH)

events_path = f"{DATASET_PATH}/events.csv"

events = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(events_path)
)

print("Columnas de events.csv:")
print(events.columns)
print("Registros cargados:", events.count())
events.show(5, truncate=False)


Ruta del dataset: /Users/guilherr/.cache/kagglehub/datasets/retailrocket/ecommerce-dataset/versions/2


Columnas de events.csv:
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']
Registros cargados: 2756101
+-------------+---------+-----+------+-------------+
|timestamp    |visitorid|event|itemid|transactionid|
+-------------+---------+-----+------+-------------+
|1433221332117|257597   |view |355908|NULL         |
|1433224214164|992329   |view |248676|NULL         |
|1433221999827|111016   |view |318965|NULL         |
|1433221955914|483717   |view |253185|NULL         |
|1433221337106|951259   |view |367447|NULL         |
+-------------+---------+-----+------+-------------+
only showing top 5 rows


In [166]:
print("Esquema original:")
events.printSchema()

null_counts = events.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in events.columns
])

print("Conteo de valores nulos por columna:")
null_counts.show(truncate=False)

print("Distribución de la variable event:")
events.groupBy("event").count().orderBy(F.desc("count")).show(truncate=False)


Esquema original:
root
 |-- timestamp: long (nullable = true)
 |-- visitorid: integer (nullable = true)
 |-- event: string (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- transactionid: integer (nullable = true)

Conteo de valores nulos por columna:


+---------+---------+-----+------+-------------+
|timestamp|visitorid|event|itemid|transactionid|
+---------+---------+-----+------+-------------+
|0        |0        |0    |0     |2733644      |
+---------+---------+-----+------+-------------+

Distribución de la variable event:
+-----------+-------+
|event      |count  |
+-----------+-------+
|view       |2664312|
|addtocart  |69332  |
|transaction|22457  |
+-----------+-------+



In [167]:
# Limpiar events.csv cambiando el formato de algunas variables
# como:
# El timestamp de RetailRocket está en milisegundos
#    -> agrupar en hora, dia de la semana, si paso en fin de semana, etc

valid_events = ["view", "addtocart", "transaction"] # Eventos validos, mantener forma

events_clean = (
    events
    .select(
        F.col("timestamp").cast("long").alias("timestamp"),
        F.col("visitorid").cast("long").alias("visitorid"),
        F.col("event").cast("string").alias("event"),
        F.col("itemid").cast("long").alias("itemid"),
        F.col("transactionid").cast("long").alias("transactionid")
    )
    .filter(F.col("event").isin(valid_events))
    .dropna(subset=["timestamp", "visitorid", "event", "itemid"])
) # Limpiar y estandarizar columnas principales

In [168]:
events_clean.show(truncate=False)

+-------------+---------+---------+------+-------------+
|timestamp    |visitorid|event    |itemid|transactionid|
+-------------+---------+---------+------+-------------+
|1433221332117|257597   |view     |355908|NULL         |
|1433224214164|992329   |view     |248676|NULL         |
|1433221999827|111016   |view     |318965|NULL         |
|1433221955914|483717   |view     |253185|NULL         |
|1433221337106|951259   |view     |367447|NULL         |
|1433224086234|972639   |view     |22556 |NULL         |
|1433221923240|810725   |view     |443030|NULL         |
|1433223291897|794181   |view     |439202|NULL         |
|1433220899221|824915   |view     |428805|NULL         |
|1433221204592|339335   |view     |82389 |NULL         |
|1433222162373|176446   |view     |10572 |NULL         |
|1433221701252|929206   |view     |410676|NULL         |
|1433224229496|15795    |view     |44872 |NULL         |
|1433223697356|598426   |view     |156489|NULL         |
|1433224078165|223343   |view  

In [169]:
events_enriched = (
    events_clean
    .withColumn("timestamp_s", (F.col("timestamp") / F.lit(1000)).cast("long"))
    .withColumn("event_ts", F.to_timestamp(F.from_unixtime(F.col("timestamp_s"))))
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("event_hour", F.hour("event_ts").cast("int"))
    .withColumn("event_dayofweek", F.dayofweek("event_ts").cast("int"))
    .withColumn("event_month", F.month("event_ts").cast("int"))
    .withColumn(
        "is_weekend",
        F.when(F.col("event_dayofweek").isin([1, 7]), F.lit(1)).otherwise(F.lit(0)).cast("int")
    )
) # formato cambiado, tipos de datos agrupados y estandarizados

events_enriched.cache()
print("Registros válidos después de limpieza básica:", events_enriched.count())
events_enriched.select(
    "timestamp", "event_ts", "visitorid", "event", "itemid", "transactionid",
    "event_hour", "event_dayofweek", "event_month", "is_weekend"
).show(10, truncate=False)


Registros válidos después de limpieza básica: 2756101
+-------------+-------------------+---------+-----+------+-------------+----------+---------------+-----------+----------+
|timestamp    |event_ts           |visitorid|event|itemid|transactionid|event_hour|event_dayofweek|event_month|is_weekend|
+-------------+-------------------+---------+-----+------+-------------+----------+---------------+-----------+----------+
|1433221332117|2015-06-02 00:02:12|257597   |view |355908|NULL         |0         |3              |6          |0         |
|1433224214164|2015-06-02 00:50:14|992329   |view |248676|NULL         |0         |3              |6          |0         |
|1433221999827|2015-06-02 00:13:19|111016   |view |318965|NULL         |0         |3              |6          |0         |
|1433221955914|2015-06-02 00:12:35|483717   |view |253185|NULL         |0         |3              |6          |0         |
|1433221337106|2015-06-02 00:02:17|951259   |view |367447|NULL         |0         |3 

26/06/01 15:58:16 WARN CacheManager: Asked to cache already cached data.


In [170]:
# Construcción de variables para particionamiento
#
# events_per_user: actividad del visitante.
# item_popularity: popularidad global del producto

user_activity = (
    events_enriched
    .groupBy("visitorid")
    .agg(F.count("*").alias("events_per_user"))
)

In [171]:
user_activity.show(10, truncate=False)

+---------+---------------+
|visitorid|events_per_user|
+---------+---------------+
|929206   |3              |
|15795    |3              |
|1377281  |1              |
|629333   |6              |
|432882   |4              |
|794013   |2              |
|741702   |1              |
|1143908  |3              |
|376913   |6              |
|1262470  |5              |
+---------+---------------+
only showing top 10 rows


In [172]:
# Se usan cuantiles 60 % y 90 %. En este dataset muchos usuarios tienen un solo evento;
# por eso se agrega una corrección por si los cuantiles quedan repetidos.
q_low, q_high = user_activity.approxQuantile("events_per_user", [0.60, 0.90], 0.01)
q_low = float(q_low)
q_high = float(q_high)
if q_high <= q_low:
    q_high = q_low + 1.0

print("Umbral actividad baja-media:", q_low)
print("Umbral actividad media-alta:", q_high)

Umbral actividad baja-media: 1.0
Umbral actividad media-alta: 3.0


In [173]:
user_activity = (
    user_activity
    .withColumn(
        "user_activity_level",
        F.when(F.col("events_per_user") <= F.lit(q_low), F.lit("baja"))
         .when(F.col("events_per_user") <= F.lit(q_high), F.lit("media"))
         .otherwise(F.lit("alta"))
    )
)
user_activity.show(10, truncate=False)

+---------+---------------+-------------------+
|visitorid|events_per_user|user_activity_level|
+---------+---------------+-------------------+
|929206   |3              |media              |
|15795    |3              |media              |
|1377281  |1              |baja               |
|629333   |6              |alta               |
|432882   |4              |alta               |
|794013   |2              |media              |
|741702   |1              |baja               |
|1143908  |3              |media              |
|376913   |6              |alta               |
|1262470  |5              |alta               |
+---------+---------------+-------------------+
only showing top 10 rows


In [174]:
item_popularity = (
    events_enriched
    .groupBy("itemid")
    .agg(F.count("*").alias("item_popularity"))
)
item_popularity.show(10, truncate=False)

+------+---------------+
|itemid|item_popularity|
+------+---------------+
|355908|57             |
|367447|260            |
|22556 |3              |
|82389 |541            |
|410676|8              |
|251467|4              |
|135256|3              |
|22926 |343            |
|216707|5              |
|386527|11             |
+------+---------------+
only showing top 10 rows


In [175]:
partitioned = (
    events_enriched
    .join(user_activity, on="visitorid", how="left")
    .join(item_popularity, on="itemid", how="left")
    .fillna({"events_per_user": 1, "item_popularity": 1, "user_activity_level": "baja"})
    .withColumn(
        "partition_name",
        F.when((F.col("event") == "view") & (F.col("user_activity_level") == "baja"), F.lit("P1"))
         .when((F.col("event") == "view") & (F.col("user_activity_level") == "media"), F.lit("P2"))
         .when((F.col("event") == "view") & (F.col("user_activity_level") == "alta"), F.lit("P3"))
         .when(F.col("event") == "addtocart", F.lit("P4"))
         .when(F.col("event") == "transaction", F.lit("P5"))
         .otherwise(F.lit("otro"))
    )
    .filter(F.col("partition_name").isin(["P1", "P2", "P3", "P4", "P5"]))
)

partitioned.cache()
print("Tamaño por partición antes del muestreo:")
partitioned.groupBy("partition_name", "event", "user_activity_level").count().orderBy("partition_name").show(50, truncate=False)


Tamaño por partición antes del muestreo:


26/06/01 15:58:18 WARN CacheManager: Asked to cache already cached data.


+--------------+-----------+-------------------+-------+
|partition_name|event      |user_activity_level|count  |
+--------------+-----------+-------------------+-------+
|P1            |view       |baja               |998953 |
|P2            |view       |media              |634729 |
|P3            |view       |alta               |1030630|
|P4            |addtocart  |alta               |53235  |
|P4            |addtocart  |media              |13560  |
|P4            |addtocart  |baja               |2537   |
|P5            |transaction|media              |2531   |
|P5            |transaction|baja               |70     |
|P5            |transaction|alta               |19856  |
+--------------+-----------+-------------------+-------+



In [176]:
# Muestreo estratificado para construir la muestra M
#
# Fracciones propuestas con base en la actividad anterior:
# - P1, P2, P3: menor fracción porque view es la clase mayoritaria.
# - P4: mayor fracción por ser señal de intención de compra.
# - P5: censo completo por ser señal escasa y valiosa.

sample_fractions_by_partition = {
    "P1": 0.02,
    "P2": 0.03,
    "P3": 0.05,
    "P4": 0.30,
    "P5": 1.00,
}

M = (
    partitioned
    .sampleBy("partition_name", fractions=sample_fractions_by_partition, seed=SEED)
    .cache()
)

print("Tamaño total de la muestra M:", M.count())
print("Distribución de M por partición y evento:")
M.groupBy("partition_name", "event").count().orderBy("partition_name", "event").show(50, truncate=False)


Tamaño total de la muestra M: 133873
Distribución de M por partición y evento:
+--------------+-----------+-----+
|partition_name|event      |count|
+--------------+-----------+-----+
|P1            |view       |19917|
|P2            |view       |19135|
|P3            |view       |51439|
|P4            |addtocart  |20925|
|P5            |transaction|22457|
+--------------+-----------+-----+



In [177]:
print("Distribución de M por variable objetivo candidata:")
M.withColumn("target_tmp", F.when(F.col("event") == "view", 0).otherwise(1)) \
 .groupBy("target_tmp", "event").count().orderBy("target_tmp", "event").show(truncate=False)

Distribución de M por variable objetivo candidata:
+----------+-----------+-----+
|target_tmp|event      |count|
+----------+-----------+-----+
|0         |view       |90491|
|1         |addtocart  |20925|
|1         |transaction|22457|
+----------+-----------+-----+



## 3. Preparación de los datos

La preparación se aplica sobre la muestra **M**. Las decisiones principales son:

- Se eliminan registros duplicados exactos en las claves del evento.
- Se validan columnas esenciales: `timestamp`, `visitorid`, `event`, `itemid`.
- `transactionid` se conserva para trazabilidad, pero **no se usa como predictor**, porque está casi completamente asociado a compras.
- Se corrigen valores extremos de `events_per_user` e `item_popularity` usando winsorización al percentil 99.
- Se crea `label`, la variable objetivo supervisada: `0 = view`; `1 = addtocart o transaction`.
- Se transforman las variables predictoras a tipo numérico `double` para ser usadas por `VectorAssembler`.


In [178]:
# Corrección de nulos, duplicados, tipos y valores atipicos
key_columns = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
required_columns = ["timestamp", "visitorid", "event", "itemid", "event_hour", "event_dayofweek", "event_month"]

print("Nulos en M antes de preparar:")
M.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in M.columns
]).show(truncate=False)

Nulos en M antes de preparar:
+------+---------+---------+-----+-------------+-----------+--------+----------+----------+---------------+-----------+----------+---------------+-------------------+---------------+--------------+
|itemid|visitorid|timestamp|event|transactionid|timestamp_s|event_ts|event_date|event_hour|event_dayofweek|event_month|is_weekend|events_per_user|user_activity_level|item_popularity|partition_name|
+------+---------+---------+-----+-------------+-----------+--------+----------+----------+---------------+-----------+----------+---------------+-------------------+---------------+--------------+
|0     |0        |0        |0    |111416       |0          |0       |0         |0         |0              |0          |0         |0              |0                  |0              |0             |
+------+---------+---------+-----+-------------+-----------+--------+----------+----------+---------------+-----------+----------+---------------+-------------------+------------

In [179]:
print("M despues de limpiar nulos:")
M_clean = (
    M
    .dropDuplicates(key_columns)
    .dropna(subset=required_columns)
    .filter(F.col("event").isin(valid_events))
)
M_clean.show(truncate=False)

M despues de limpiar nulos:
+------+---------+-------------+-----------+-------------+-----------+-------------------+----------+----------+---------------+-----------+----------+---------------+-------------------+---------------+--------------+
|itemid|visitorid|timestamp    |event      |transactionid|timestamp_s|event_ts           |event_date|event_hour|event_dayofweek|event_month|is_weekend|events_per_user|user_activity_level|item_popularity|partition_name|
+------+---------+-------------+-----------+-------------+-----------+-------------------+----------+----------+---------------+-----------+----------+---------------+-------------------+---------------+--------------+
|62950 |340221   |1430622284504|view       |NULL         |1430622284 |2015-05-02 22:04:44|2015-05-02|22        |7              |5          |1         |4              |alta               |10             |P3            |
|461547|1375898  |1430622912969|addtocart  |NULL         |1430622912 |2015-05-02 22:15:12|2015-0

In [180]:
# Percentiles para limitar atipicos. Si una columna tiene valores altos,
# se reemplaza por el p99 para reducir dominancia de usuarios/productos muy frecuentes
p99_user = M_clean.approxQuantile("events_per_user", [0.99], 0.01)[0]
p99_item = M_clean.approxQuantile("item_popularity", [0.99], 0.01)[0]

p99_user = float(p99_user) if p99_user is not None else 1.0
p99_item = float(p99_item) if p99_item is not None else 1.0

print("p99 events_per_user:", p99_user)
print("p99 item_popularity:", p99_item)

p99 events_per_user: 7757.0
p99 item_popularity: 3412.0


In [181]:
M_prepared = (
    M_clean
    .withColumn("events_per_user_capped", F.least(F.col("events_per_user").cast("double"), F.lit(p99_user)))
    .withColumn("item_popularity_capped", F.least(F.col("item_popularity").cast("double"), F.lit(p99_item)))
    .withColumn("event_hour", F.col("event_hour").cast("double"))
    .withColumn("event_dayofweek", F.col("event_dayofweek").cast("double"))
    .withColumn("event_month", F.col("event_month").cast("double"))
    .withColumn("is_weekend", F.col("is_weekend").cast("double"))
    .withColumn("label", F.when(F.col("event") == "view", F.lit(0.0)).otherwise(F.lit(1.0)))
    .cache()
)

feature_cols = [
    "event_hour",
    "event_dayofweek",
    "event_month",
    "is_weekend",
    "events_per_user_capped",
    "item_popularity_capped",
]

print("Tamaño de M después de preparación:", M_prepared.count())
print("Distribución de la variable objetivo label:")
M_prepared.groupBy("label", "event").count().orderBy("label", "event").show(truncate=False)

print("Vista de columnas preparadas:")
M_prepared.select(feature_cols + ["label", "event", "partition_name", "visitorid", "itemid"]).show(10, truncate=False)

Tamaño de M después de preparación: 133837
Distribución de la variable objetivo label:
+-----+-----------+-----+
|label|event      |count|
+-----+-----------+-----+
|0.0  |view       |90491|
|1.0  |addtocart  |20889|
|1.0  |transaction|22457|
+-----+-----------+-----+

Vista de columnas preparadas:
+----------+---------------+-----------+----------+----------------------+----------------------+-----+-----------+--------------+---------+------+
|event_hour|event_dayofweek|event_month|is_weekend|events_per_user_capped|item_popularity_capped|label|event      |partition_name|visitorid|itemid|
+----------+---------------+-----------+----------+----------------------+----------------------+-----+-----------+--------------+---------+------+
|22.0      |7.0            |5.0        |1.0       |4.0                   |10.0                  |0.0  |view       |P3            |340221   |62950 |
|22.0      |7.0            |5.0        |1.0       |22.0                  |38.0                  |1.0  |addto

## 4. Preparación del conjunto de entrenamiento y prueba

Para minimizar inyección de sesgos y fuga de información, la división se realiza por `visitorid`: todos los eventos de un mismo visitante quedan en entrenamiento o en prueba, pero no en ambos. Esta estrategia es preferible a dividir fila por fila, porque los usuarios pueden tener múltiples eventos y el modelo podría memorizar patrones individuales si sus eventos aparecen en ambos conjuntos.

Se usa una partición determinística por hash:

- 80 % de visitantes para entrenamiento.
- 20 % de visitantes para prueba.


In [182]:
# Separar train/test por visitante para evitar fuga entre conjuntos

visitor_split = (
    M_prepared
    .select("visitorid")
    .distinct()
    .withColumn("split_bucket", F.pmod(F.hash(F.col("visitorid")), F.lit(100)))
    .withColumn("dataset_split", F.when(F.col("split_bucket") < F.lit(80), F.lit("train")).otherwise(F.lit("test")))
    .select("visitorid", "dataset_split")
)

M_split = M_prepared.join(visitor_split, on="visitorid", how="left")

train_df = M_split.filter(F.col("dataset_split") == "train").drop("dataset_split", "split_bucket").cache()
test_df = M_split.filter(F.col("dataset_split") == "test").drop("dataset_split", "split_bucket").cache()

print("Registros train:", train_df.count())
print("Registros test:", test_df.count())


Registros train: 107472
Registros test: 26365


In [183]:
print("Distribución de label en train:")
train_df.groupBy("label").count().orderBy("label").show()

print("Distribución de label en test:")
test_df.groupBy("label").count().orderBy("label").show()

# Verificación explícita de fuga de usuarios entre train y test.
visitors_in_both = train_df.select("visitorid").distinct().intersect(
    test_df.select("visitorid").distinct()
).count()

print("Visitantes presentes en train y test simultáneamente:", visitors_in_both)

Distribución de label en train:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|72395|
|  1.0|35077|
+-----+-----+

Distribución de label en test:
+-----+-----+
|label|count|
+-----+-----+
|  0.0|18096|
|  1.0| 8269|
+-----+-----+

Visitantes presentes en train y test simultáneamente: 0


In [184]:
# =========================================================
# 10) Importancia de variables e interpretación del modelo supervisado
# =========================================================

rf_stage = rf_model.stages[-1]
feature_importance_rows = [
    (feature_cols[i], float(importance))
    for i, importance in enumerate(rf_stage.featureImportances)
]

print("Importancia de variables según Random Forest:")
spark.createDataFrame(feature_importance_rows, ["feature", "importance"]) \
    .orderBy(F.desc("importance")) \
    .show(truncate=False)

print("Discusión automática del experimento supervisado:")
print(f"- La tasa positiva en test fue {base_rate:.4f}; esta es una línea base para interpretar areaUnderPR.")
print(f"- AUC ROC = {auc_roc:.4f}. Valores cercanos a 1 indican mejor separación entre eventos view y eventos de intención/conversión.")
print(f"- AUC PR = {auc_pr:.4f}. Si supera claramente la tasa positiva base, el modelo aporta valor frente a una clasificación aleatoria.")
print(f"- F1 = {f1_score:.4f}; resume equilibrio entre precisión y recall ponderados.")
print("- transactionid, visitorid e itemid no se usaron como variables predictoras directas para reducir fuga de información y memorización.")


Importancia de variables según Random Forest:
+----------------------+--------------------+
|feature               |importance          |
+----------------------+--------------------+
|events_per_user_capped|0.7853437780976149  |
|item_popularity_capped|0.16821867341173904 |
|event_hour            |0.022490222853308525|
|event_month           |0.009517736559762546|
|event_dayofweek       |0.007979964648352902|
|is_weekend            |0.006449624429222245|
+----------------------+--------------------+

Discusión automática del experimento supervisado:
- La tasa positiva en test fue 0.3136; esta es una línea base para interpretar areaUnderPR.
- AUC ROC = 0.7202. Valores cercanos a 1 indican mejor separación entre eventos view y eventos de intención/conversión.
- AUC PR = 0.4931. Si supera claramente la tasa positiva base, el modelo aporta valor frente a una clasificación aleatoria.
- F1 = 0.6013; resume equilibrio entre precisión y recall ponderados.
- transactionid, visitorid e itemid n

## 5. Construcción del modelo de aprendizaje supervisado

### Variable objetivo

La columna objetivo es `label`:

- `label = 0.0`: evento `view`.
- `label = 1.0`: evento `addtocart` o `transaction`.

El problema se formula como una clasificación binaria para identificar eventos de mayor intención comercial. Las variables de entrada seleccionadas son temporales y conductuales:

`event_hour`, `event_dayofweek`, `event_month`, `is_weekend`, `events_per_user_capped`, `item_popularity_capped`.

### Algoritmo seleccionado

Se usa `RandomForestClassifier`, un ensamble de árboles de decisión. Para medir la calidad del resultado se usan:

- `areaUnderROC`: capacidad general de discriminación.
- `areaUnderPR`: más informativa cuando las clases están desbalanceadas.
- `accuracy` y `f1`: rendimiento global de clasificación.
- Matriz de confusión: errores por clase.


In [185]:
# Modelo supervisado: RandomForestClassifier

# Validación mínima: se requieren al menos dos clases en train y test.
if train_df.select("label").distinct().count() < 2:
    raise ValueError("El conjunto de entrenamiento tiene una sola clase. Ajusta las fracciones de muestreo.")
if test_df.select("label").distinct().count() < 2:
    raise ValueError("El conjunto de prueba tiene una sola clase. Ajusta las fracciones de muestreo o el split.")

In [186]:
# Peso por clase para compensar desbalance dentro de la muestra de entrenamiento.
label_counts = train_df.groupBy("label").count().collect()
total_train = sum(row["count"] for row in label_counts)
num_classes = len(label_counts)
weight_rows = [
    (float(row["label"]), float(total_train) / (num_classes * float(row["count"])))
    for row in label_counts
]
weights_df = spark.createDataFrame(weight_rows, ["label", "class_weight"])

train_weighted = (
    train_df
    .join(F.broadcast(weights_df), on="label", how="left")
    .fillna({"class_weight": 1.0})
    .cache()
)

print("Pesos por clase:")
weights_df.orderBy("label").show()

Pesos por clase:
+-----+------------------+
|label|      class_weight|
+-----+------------------+
|  0.0|0.7422612058843843|
|  1.0|1.5319440088947174|
+-----+------------------+



In [187]:
assembler_supervised = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_supervised",
    handleInvalid="keep"
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features_supervised",
    weightCol="class_weight",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=80,
    maxDepth=8,
    seed=SEED
)

rf_pipeline = Pipeline(stages=[assembler_supervised, rf])
rf_model = rf_pipeline.fit(train_weighted)

pred_rf = rf_model.transform(test_df).cache()

26/06/01 15:58:26 WARN DAGScheduler: Broadcasting large task binary with size 1596.8 KiB
26/06/01 15:58:27 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
                                                                                

In [188]:
# Métricas de evaluación.
binary_roc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
binary_pr = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
precision_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)

auc_roc = binary_roc.evaluate(pred_rf)
auc_pr = binary_pr.evaluate(pred_rf)
accuracy = accuracy_eval.evaluate(pred_rf)
f1_score = f1_eval.evaluate(pred_rf)
weighted_precision = precision_eval.evaluate(pred_rf)
weighted_recall = recall_eval.evaluate(pred_rf)
base_rate = test_df.agg(F.avg("label").alias("positive_rate")).first()["positive_rate"]

metrics = [
    ("positive_rate_test", float(base_rate)),
    ("areaUnderROC", float(auc_roc)),
    ("areaUnderPR", float(auc_pr)),
    ("accuracy", float(accuracy)),
    ("f1", float(f1_score)),
    ("weightedPrecision", float(weighted_precision)),
    ("weightedRecall", float(weighted_recall)),
]

spark.createDataFrame(metrics, ["metric", "value"]).show(truncate=False)

print("Matriz de confusión:")
pred_rf.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

print("Predicciones de ejemplo:")
pred_rf.select("event", "label", "prediction", "probability", *feature_cols).show(10, truncate=False)

26/06/01 15:58:29 WARN DAGScheduler: Broadcasting large task binary with size 1567.0 KiB
26/06/01 15:58:29 WARN DAGScheduler: Broadcasting large task binary with size 1581.7 KiB
26/06/01 15:58:29 WARN DAGScheduler: Broadcasting large task binary with size 1581.7 KiB
26/06/01 15:58:30 WARN DAGScheduler: Broadcasting large task binary with size 1581.8 KiB
26/06/01 15:58:30 WARN DAGScheduler: Broadcasting large task binary with size 1581.8 KiB
26/06/01 15:58:30 WARN DAGScheduler: Broadcasting large task binary with size 1581.8 KiB
26/06/01 15:58:30 WARN DAGScheduler: Broadcasting large task binary with size 1581.8 KiB


+------------------+------------------+
|metric            |value             |
+------------------+------------------+
|positive_rate_test|0.3136355016119856|
|areaUnderROC      |0.7201846230351898|
|areaUnderPR       |0.4930972952854632|
|accuracy          |0.593893419305898 |
|f1                |0.6013189377529   |
|weightedPrecision |0.7358113713882657|
|weightedRecall    |0.593893419305898 |
+------------------+------------------+

Matriz de confusión:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0| 8592|
|  0.0|       1.0| 9504|
|  1.0|       0.0| 1203|
|  1.0|       1.0| 7066|
+-----+----------+-----+

Predicciones de ejemplo:
+-----------+-----+----------+----------------------------------------+----------+---------------+-----------+----------+----------------------+----------------------+
|event      |label|prediction|probability                             |event_hour|event_dayofweek|event_month|is_weekend|events_per_user_capped|

26/06/01 15:58:30 WARN DAGScheduler: Broadcasting large task binary with size 1583.4 KiB
26/06/01 15:58:30 WARN DAGScheduler: Broadcasting large task binary with size 1574.4 KiB


In [189]:
# Valores e interpretacion de resultados

rf_stage = rf_model.stages[-1]
feature_importance_rows = [
    (feature_cols[i], float(importance))
    for i, importance in enumerate(rf_stage.featureImportances)
]

print("Importancia de variables según Random Forest:")
spark.createDataFrame(feature_importance_rows, ["feature", "importance"]) \
    .orderBy(F.desc("importance")) \
    .show(truncate=False)

Importancia de variables según Random Forest:
+----------------------+--------------------+
|feature               |importance          |
+----------------------+--------------------+
|events_per_user_capped|0.7853437780976149  |
|item_popularity_capped|0.16821867341173904 |
|event_hour            |0.022490222853308525|
|event_month           |0.009517736559762546|
|event_dayofweek       |0.007979964648352902|
|is_weekend            |0.006449624429222245|
+----------------------+--------------------+



In [190]:
print(f"- La tasa positiva en test fue {base_rate:.4f}; esta es una línea base para interpretar areaUnderPR.")
print(f"- AUC ROC = {auc_roc:.4f}. Valores cercanos a 1 indican mejor separación entre eventos view y eventos de intención/conversión.")
print(f"- AUC PR = {auc_pr:.4f}. Si supera claramente la tasa positiva base, el modelo aporta valor frente a una clasificación aleatoria.")
print(f"- F1 = {f1_score:.4f}; resume equilibrio entre precisión y recall ponderados.")
print("- transactionid, visitorid e itemid no se usaron como variables predictoras directas para reducir fuga de información y memorización.")

- La tasa positiva en test fue 0.3136; esta es una línea base para interpretar areaUnderPR.
- AUC ROC = 0.7202. Valores cercanos a 1 indican mejor separación entre eventos view y eventos de intención/conversión.
- AUC PR = 0.4931. Si supera claramente la tasa positiva base, el modelo aporta valor frente a una clasificación aleatoria.
- F1 = 0.6013; resume equilibrio entre precisión y recall ponderados.
- transactionid, visitorid e itemid no se usaron como variables predictoras directas para reducir fuga de información y memorización.


## 6. Construcción del modelo de aprendizaje no supervisado

Para el experimento no supervisado se usa `KMeans`. El objetivo es identificar grupos de eventos con comportamientos similares, sin entregar la etiqueta `label` al algoritmo.

Las variables usadas son las mismas variables numéricas de comportamiento y temporalidad:

`event_hour`, `event_dayofweek`, `event_month`, `is_weekend`, `events_per_user_capped`, `item_popularity_capped`.

Se estandarizan las variables con `StandardScaler`, porque K-Means depende de distancias y puede verse afectado por escalas distintas. Se prueban valores de `k` entre 2 y 6, y se selecciona el mejor valor según el índice de **silhouette** (basado en el siguiente doc https://medium.com/@nicolasarrioja/c%C3%B3mo-seleccionar-el-mejor-valor-de-k-en-k-means-21121b604365)


In [191]:
# Modelo no supervisado: KMeans con seleccion de K basado en el indice silhouette

assembler_unsupervised = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_unsupervised",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features_unsupervised",
    outputCol="features_unsupervised_scaled",
    withMean=True,
    withStd=True
)

cluster_evaluator = ClusteringEvaluator(
    featuresCol="features_unsupervised_scaled",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

In [192]:
k_values = [2, 3, 4, 5, 6]
k_results = []
k_models = {}

for k in k_values:
    kmeans = KMeans(
        k=k,
        seed=SEED,
        featuresCol="features_unsupervised_scaled",
        predictionCol="cluster",
        maxIter=30,
        initSteps=5
    )
    kmeans_pipeline = Pipeline(stages=[assembler_unsupervised, scaler, kmeans])
    model = kmeans_pipeline.fit(train_df)
    pred_k = model.transform(test_df)
    silhouette = cluster_evaluator.evaluate(pred_k)
    k_results.append((k, float(silhouette)))
    k_models[k] = model

k_metrics_df = spark.createDataFrame(k_results, ["k", "silhouette"])
print("Silhouette por número de clusters:")
k_metrics_df.orderBy(F.desc("silhouette")).show()

Silhouette por número de clusters:
+---+-------------------+
|  k|         silhouette|
+---+-------------------+
|  4| 0.5026925558460398|
|  6|0.43371314846259074|
|  5| 0.2784474785468258|
|  2|0.25552536399349557|
|  3|0.24140513719484324|
+---+-------------------+



In [193]:
best_k, best_silhouette = max(k_results, key=lambda row: row[1])
best_kmeans_model = k_models[best_k]

print(f"Mejor k según silhouette: {best_k}")
print(f"Silhouette test del mejor k: {best_silhouette:.4f}")


Mejor k según silhouette: 4
Silhouette test del mejor k: 0.5027


In [194]:
clustered = best_kmeans_model.transform(M_prepared).cache()
print("Tamaño de cada cluster:")
clustered.groupBy("cluster").count().orderBy("cluster").show()

Tamaño de cada cluster:
+-------+-----+
|cluster|count|
+-------+-----+
|      0| 4325|
|      1|98910|
|      2|28947|
|      3| 1655|
+-------+-----+



In [195]:
# Perfilamiento de clusters
# Aunque label/event no se usaron para entrenar KMeans, se usan aquí para interpretar los grupos

cluster_profile = (
    clustered
    .groupBy("cluster")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("event_hour"), 2).alias("avg_event_hour"),
        F.round(F.avg("event_dayofweek"), 2).alias("avg_dayofweek"),
        F.round(F.avg("is_weekend"), 4).alias("prop_weekend"),
        F.round(F.avg("events_per_user_capped"), 2).alias("avg_events_per_user_capped"),
        F.round(F.avg("item_popularity_capped"), 2).alias("avg_item_popularity_capped"),
        F.round(F.avg("label"), 4).alias("prop_intencion_conversion")
    )
    .orderBy("cluster")
)

print("Perfil numérico de clusters:")
cluster_profile.show(truncate=False)

Perfil numérico de clusters:
+-------+-----+--------------+-------------+------------+--------------------------+--------------------------+-------------------------+
|cluster|n    |avg_event_hour|avg_dayofweek|prop_weekend|avg_events_per_user_capped|avg_item_popularity_capped|prop_intencion_conversion|
+-------+-----+--------------+-------------+------------+--------------------------+--------------------------+-------------------------+
|0      |4325 |14.19         |3.8          |0.1773      |155.04                    |1345.43                   |0.4296                   |
|1      |98910|14.12         |3.89         |0.0         |132.78                    |102.43                    |0.3257                   |
|2      |28947|14.26         |3.74         |1.0         |72.67                     |106.78                    |0.2795                   |
|3      |1655 |14.87         |4.11         |0.2054      |6551.15                   |164.49                    |0.7166                   |
+----

In [196]:
print("Distribución de eventos por cluster:")
clustered.groupBy("cluster").pivot("event").count().na.fill(0).orderBy("cluster").show(truncate=False)

cluster_totals = clustered.groupBy("cluster").agg(F.count("*").alias("cluster_n"))
cluster_event_props = (
    clustered
    .groupBy("cluster", "event")
    .agg(F.count("*").alias("event_n"))
    .join(cluster_totals, on="cluster", how="left")
    .withColumn("event_prop_in_cluster", F.round(F.col("event_n") / F.col("cluster_n"), 4))
    .orderBy("cluster", "event")
)

print("Proporción de cada evento dentro de cada cluster:")
cluster_event_props.show(100, truncate=False)

Distribución de eventos por cluster:
+-------+---------+-----------+-----+
|cluster|addtocart|transaction|view |
+-------+---------+-----------+-----+
|0      |879      |979        |2467 |
|1      |15301    |16910      |66699|
|2      |4365     |3726       |20856|
|3      |344      |842        |469  |
+-------+---------+-----------+-----+

Proporción de cada evento dentro de cada cluster:
+-------+-----------+-------+---------+---------------------+
|cluster|event      |event_n|cluster_n|event_prop_in_cluster|
+-------+-----------+-------+---------+---------------------+
|0      |addtocart  |879    |4325     |0.2032               |
|0      |transaction|979    |4325     |0.2264               |
|0      |view       |2467   |4325     |0.5704               |
|1      |addtocart  |15301  |98910    |0.1547               |
|1      |transaction|16910  |98910    |0.171                |
|1      |view       |66699  |98910    |0.6743               |
|2      |addtocart  |4365   |28947    |0.1508     

In [197]:
print(f"- El mejor número de grupos probado fue k={best_k}, con silhouette={best_silhouette:.4f} en test.")

- El mejor número de grupos probado fue k=4, con silhouette=0.5027 en test.


### La separación entre clusters es razonablemente alta según silhouette (> .50)

## 7. Conclusiones de la entrega

1. Se documento la diferencia entre aprendizaje supervisado y no supervisado, junto con algunos algoritmos representativos disponibles en PySpark
2. Se construyó una muestra **M** desde `events.csv` usando particiones por tipo de evento y nivel de actividad del usuario.
3. Se prepararon los datos corrigiendo nulos, duplicados, tipos de dato y valores atípicos.
4. Se dividió la muestra en entrenamiento y prueba por `visitorid`, reduciendo el riesgo de que eventos del mismo usuario aparezcan en ambos conjuntos.
5. Se entrenó un modelo supervisado `RandomForestClassifier` para predecir intención/conversión, evaluado con AUC ROC, AUC PR, F1 y matriz de confusión.
6. Se entrenó un modelo no supervisado `KMeans`, evaluado con silhouette y perfilado por cluster.
